# GRPO + vLLM Final Validation and Performance Baseline

This notebook is the **final confirmation flow** for training signal + rollout weight-sync behavior.

You will validate:

1. **Training-side learning evidence** — GRPO `TrainJob` probe or minimal in-process GRPO run that emits reward/loss trend.
2. **Baseline BEFORE weight sync** — rollout latency/throughput with initial weights.
3. **Deterministic weight-sync validation** — unsynced to synced transition proof.
4. **Baseline AFTER weight sync** — rollout performance after sync.
5. **Rollout control-plane health** — `/health`, `/get_world_size`, `/pause`, `/resume`.


## Runtime auto-detection (usually no edits)

This notebook is designed to run directly in an OpenShift AI workbench with minimal setup.

Auto-detected by default:

- namespace (`/var/run/secrets/kubernetes.io/serviceaccount/namespace`, then `NOTEBOOK_NAMESPACE`, then `oc project -q`)
- workbench pod (`HOSTNAME` when in-pod, otherwise the newest running workbench-labeled pod in the namespace)
- notebook container (first container in the selected workbench pod)

Optional overrides are available in the next cell for advanced or out-of-cluster runs, but most runs should be plug-and-play.


In [ ]:
from __future__ import annotations

import datetime as dt
import json
import os
import re
import shlex
import subprocess
import time
from pathlib import Path
from typing import Any


def find_grpo_root() -> Path:
    env_root = os.getenv('GRPO_TRAINER_ROOT')
    search_roots: list[Path] = []
    if env_root:
        search_roots.append(Path(env_root).expanduser().resolve())

    cwd = Path.cwd().resolve()
    search_roots.extend([cwd, *cwd.parents])

    for base in search_roots:
        direct_validate = base / 'scripts' / 'validate_cold_start_weight_sync.sh'
        direct_baseline = base / 'scripts' / 'capture_vllm_performance_baseline.sh'
        if direct_validate.exists() and direct_baseline.exists():
            return base

        nested = base / 'research' / 'grpo-trainer'
        nested_validate = nested / 'scripts' / 'validate_cold_start_weight_sync.sh'
        nested_baseline = nested / 'scripts' / 'capture_vllm_performance_baseline.sh'
        if nested_validate.exists() and nested_baseline.exists():
            return nested

    raise FileNotFoundError(
        'Could not locate grpo-trainer root. Set GRPO_TRAINER_ROOT or run this notebook from inside the repo.'
    )


ROOT = find_grpo_root()
SCRIPT_VALIDATE = ROOT / 'scripts' / 'validate_cold_start_weight_sync.sh'
SCRIPT_BASELINE = ROOT / 'scripts' / 'capture_vllm_performance_baseline.sh'

RUN_TS = dt.datetime.now().strftime('%Y%m%d-%H%M%S')
OUTPUT_ROOT = ROOT / 'runs_log' / f'notebook-final-validation-{RUN_TS}'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Research root: {ROOT}')
print(f'Run output dir: {OUTPUT_ROOT}')


def run(cmd: str, *, check: bool = True, env: dict | None = None):
    merged_env = os.environ.copy()
    if env:
        merged_env.update({k: str(v) for k, v in env.items()})
    print(f"\n$ {cmd}")
    proc = subprocess.run(
        cmd,
        shell=True,
        cwd=str(ROOT),
        text=True,
        capture_output=True,
        env=merged_env,
    )
    if proc.stdout:
        print(proc.stdout.rstrip())
    if proc.stderr:
        print(proc.stderr.rstrip())
    print(f'[exit_code={proc.returncode}]')
    if check and proc.returncode != 0:
        raise RuntimeError(f'Command failed ({proc.returncode}): {cmd}')
    return proc


def latest_file(directory: Path, pattern: str) -> Path:
    matches = sorted(directory.glob(pattern))
    if not matches:
        raise FileNotFoundError(f'No file matching {pattern} in {directory}')
    return matches[-1]


def extract_metrics(payload: dict[str, Any]) -> dict[str, Any]:
    s = payload.get('signals', {})
    return {
        'latency_ms_p50': s.get('latency_ms_p50'),
        'latency_ms_p95': s.get('latency_ms_p95'),
        'ttft_ms_p50': s.get('ttft_ms_p50'),
        'ttft_ms_p95': s.get('ttft_ms_p95'),
        'throughput_rps': s.get('throughput_rps'),
        'token_throughput_tokens_per_s': s.get('token_throughput_tokens_per_s'),
        'error_rate': s.get('error_rate'),
    }


def pct_delta(before: float | None, after: float | None):
    if before in (None, 0) or after is None:
        return None
    return ((after - before) / before) * 100.0


def detect_namespace() -> str:
    # Preferred order: service account namespace file -> NOTEBOOK_NAMESPACE -> current oc project
    sa_ns_path = Path('/var/run/secrets/kubernetes.io/serviceaccount/namespace')
    if sa_ns_path.exists():
        ns = sa_ns_path.read_text().strip()
        if ns:
            return ns

    env_ns = os.getenv('NOTEBOOK_NAMESPACE', '').strip()
    if env_ns:
        return env_ns

    proc = run('oc project -q', check=False)
    ns = (proc.stdout or '').strip()
    if proc.returncode == 0 and ns:
        return ns

    raise RuntimeError(
        'Could not auto-detect namespace. Ensure oc login/project is configured or set NOTEBOOK_NAMESPACE.'
    )


def _list_pods(namespace: str) -> list[dict[str, Any]]:
    proc = run(f'oc get pods -n {shlex.quote(namespace)} -o json')
    payload = json.loads(proc.stdout or '{}')
    return payload.get('items', [])


def detect_workbench_pod(namespace: str, override: str | None = None) -> tuple[str, str]:
    if override:
        run(f'oc get pod {shlex.quote(override)} -n {shlex.quote(namespace)}')
        return override, 'override'

    hostname = os.getenv('HOSTNAME', '').strip()
    if hostname:
        probe = run(
            f'oc get pod {shlex.quote(hostname)} -n {shlex.quote(namespace)}',
            check=False,
        )
        if probe.returncode == 0:
            return hostname, 'HOSTNAME'

    pods = _list_pods(namespace)
    running = [p for p in pods if p.get('status', {}).get('phase') == 'Running']

    def is_workbench(pod: dict[str, Any]) -> bool:
        md = pod.get('metadata', {})
        labels = md.get('labels', {}) or {}
        name = md.get('name', '').lower()
        if 'notebook-name' in labels:
            return True
        component = (labels.get('app.kubernetes.io/component') or '').lower()
        app_name = (labels.get('app.kubernetes.io/name') or '').lower()
        dashboard = labels.get('opendatahub.io/dashboard')
        if component in {'workbench', 'notebook'}:
            return True
        if app_name in {'workbench', 'notebook'}:
            return True
        if dashboard is not None:
            return True
        return 'workbench' in name or 'notebook' in name

    workbench_candidates = [p for p in running if is_workbench(p)]

    def sort_key(pod: dict[str, Any]) -> str:
        return pod.get('metadata', {}).get('creationTimestamp', '')

    if workbench_candidates:
        selected = sorted(workbench_candidates, key=sort_key, reverse=True)[0]
        return selected['metadata']['name'], 'workbench-label-discovery'

    if running:
        selected = sorted(running, key=sort_key, reverse=True)[0]
        return selected['metadata']['name'], 'running-pod-fallback'

    raise RuntimeError(f'No running pods found in namespace {namespace}.')


def detect_container(namespace: str, pod_name: str, override: str | None = None) -> str:
    if override:
        return override

    proc = run(
        f"oc get pod {shlex.quote(pod_name)} -n {shlex.quote(namespace)} -o jsonpath='{{.spec.containers[0].name}}'"
    )
    container = (proc.stdout or '').strip()
    if not container:
        raise RuntimeError(f'Could not detect container for pod {pod_name}.')
    return container


def derive_workbench_name(namespace: str, pod_name: str) -> str:
    pod = run(
        f'oc get pod {shlex.quote(pod_name)} -n {shlex.quote(namespace)} -o json',
        check=False,
    )
    if pod.returncode == 0:
        payload = json.loads(pod.stdout or '{}')
        labels = payload.get('metadata', {}).get('labels', {}) or {}
        if labels.get('notebook-name'):
            return labels['notebook-name']

    if '-' in pod_name and pod_name.rsplit('-', 1)[-1].isdigit():
        return pod_name.rsplit('-', 1)[0]
    return pod_name


In [ ]:
# Optional overrides (leave unset for auto-detect)
OVERRIDE_NAMESPACE = os.getenv('VALIDATION_NAMESPACE') or None
OVERRIDE_NOTEBOOK_POD = (
    os.getenv('VALIDATION_NOTEBOOK_POD')
    or os.getenv('WORKBENCH_POD_NAME')
    or None
)
OVERRIDE_NOTEBOOK_CONTAINER = os.getenv('VALIDATION_NOTEBOOK_CONTAINER') or None

# Rollout defaults
VLLM_SERVICE = os.getenv('VLLM_SERVICE', 'grpo-vllm-rollout')
VLLM_DEPLOYMENT = os.getenv('VLLM_DEPLOYMENT', VLLM_SERVICE)
VLLM_SELECTOR = os.getenv('VLLM_SELECTOR', f'app={VLLM_SERVICE}')
MODEL_NAME = os.getenv('MODEL_NAME', 'Qwen/Qwen2.5-0.5B-Instruct')

# Baseline and validation controls
BASELINE_REQUESTS = int(os.getenv('BASELINE_REQUESTS', '30'))
BASELINE_MAX_TOKENS = int(os.getenv('BASELINE_MAX_TOKENS', '64'))
VALIDATION_ATTEMPTS = int(os.getenv('VALIDATION_ATTEMPTS', '3'))
VALIDATION_SKIP_RESTART = os.getenv('VALIDATION_SKIP_RESTART', 'false').lower() == 'true'

# Training signal validation controls
TRAINING_JOB_NAME = (os.getenv('TRAINING_JOB_NAME') or '').strip() or None
TRAINING_JOB_NAME_REGEX = os.getenv('TRAINING_JOB_NAME_REGEX', 'grpo')
TRAINING_WAIT_TIMEOUT_S = int(os.getenv('TRAINING_WAIT_TIMEOUT_S', '600'))
TRAINING_POLL_INTERVAL_S = int(os.getenv('TRAINING_POLL_INTERVAL_S', '15'))
TRAINING_LOG_TAIL_LINES = int(os.getenv('TRAINING_LOG_TAIL_LINES', '3000'))
TRAINING_SIGNAL_REQUIRED = os.getenv('TRAINING_SIGNAL_REQUIRED', 'true').lower() == 'true'
TRAINING_ALLOW_SKIP = os.getenv('TRAINING_ALLOW_SKIP', 'false').lower() == 'true'

# Auto-resolve runtime context
NAMESPACE = OVERRIDE_NAMESPACE or detect_namespace()
NOTEBOOK_POD, NOTEBOOK_POD_SOURCE = detect_workbench_pod(NAMESPACE, OVERRIDE_NOTEBOOK_POD)
NOTEBOOK_CONTAINER = detect_container(NAMESPACE, NOTEBOOK_POD, OVERRIDE_NOTEBOOK_CONTAINER)
NOTEBOOK_NAME = derive_workbench_name(NAMESPACE, NOTEBOOK_POD)

print('Resolved namespace:', NAMESPACE)
print('Resolved notebook pod:', NOTEBOOK_POD, f'(source={NOTEBOOK_POD_SOURCE})')
print('Resolved notebook container:', NOTEBOOK_CONTAINER)
print('Resolved notebook name:', NOTEBOOK_NAME)
print('Model:', MODEL_NAME)
print('Training signal required:', TRAINING_SIGNAL_REQUIRED)
print('Training skip allowed:', TRAINING_ALLOW_SKIP)
if TRAINING_JOB_NAME:
    print('Training job override:', TRAINING_JOB_NAME)

In [ ]:
# Preflight checks + rollout readiness gate
run(f'oc project {shlex.quote(NAMESPACE)}')
run(f'oc get pod {shlex.quote(NOTEBOOK_POD)} -n {shlex.quote(NAMESPACE)}')
run(f'oc get svc {shlex.quote(VLLM_SERVICE)} -n {shlex.quote(NAMESPACE)}')
run(f'oc get deployment {shlex.quote(VLLM_DEPLOYMENT)} -n {shlex.quote(NAMESPACE)}')
selector_probe = run(
    f'oc get pods -n {shlex.quote(NAMESPACE)} -l {shlex.quote(VLLM_SELECTOR)} -o name',
    check=False,
)
if selector_probe.returncode != 0:
    print('Warning: vLLM selector probe failed. Continue if selector is intentionally custom.')

service_host = f"{VLLM_SERVICE}.{NAMESPACE}.svc.cluster.local"
service_root = f"http://{service_host}:8000"


def _probe_service(path: str) -> bool:
    url = f"{service_root}{path}"
    cmd = (
        f"oc exec -n {shlex.quote(NAMESPACE)} {shlex.quote(NOTEBOOK_POD)} "
        f"-c {shlex.quote(NOTEBOOK_CONTAINER)} -- "
        f"curl -sS -m 10 -o /dev/null -w '%{{http_code}}' {shlex.quote(url)}"
    )
    probe = run(cmd, check=False)
    code = ''.join(ch for ch in (probe.stdout or '') if ch.isdigit())
    return probe.returncode == 0 and code.endswith('200')


def wait_for_rollout_readiness(timeout_s: int = 180, interval_s: int = 10) -> bool:
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        health_ok = _probe_service('/health')
        world_ok = _probe_service('/get_world_size')
        if health_ok and world_ok:
            return True
        print('Rollout service not ready yet; retrying...')
        time.sleep(interval_s)
    return False


ready = wait_for_rollout_readiness()
if not ready:
    print('Rollout still not ready; restarting rollout deployment once...')
    run(f'oc rollout restart deployment/{shlex.quote(VLLM_DEPLOYMENT)} -n {shlex.quote(NAMESPACE)}')
    run(f'oc rollout status deployment/{shlex.quote(VLLM_DEPLOYMENT)} -n {shlex.quote(NAMESPACE)} --timeout=600s')
    ready = wait_for_rollout_readiness(timeout_s=240, interval_s=10)

if not ready:
    raise RuntimeError(
        'Rollout service is not reachable from the workbench pod after restart. '
        'Check network policy, DNS, and rollout pod logs.'
    )

print('Notebook context is ready for baseline + validation.')
print('Service root:', service_root)
print('Scripts present:', SCRIPT_VALIDATE.exists(), SCRIPT_BASELINE.exists())


## 1) Training-side learning signal (GRPO)

This section provides **concrete evidence of a GRPO training step**: reward trend, loss trend, and total runtime.

**Two modes (auto-selected by `TRAINING_MODE` env var):**

| Mode | When to use | What it does |
|------|-------------|--------------|
| `trainjob` | Kubeflow Trainer v2 available in cluster | Submits a lightweight 1-epoch GRPO `TrainJob`, polls status, scrapes `SPIKE_SUMMARY_JSON` from logs |
| `inline` | No KFT v2 / demo on CPU | Runs a micro GRPO step in-process (10 GSM8K examples, 1 epoch, CPU-safe) and prints reward/loss |

Set `TRAINING_MODE=trainjob` or `TRAINING_MODE=inline` to override; default is `trainjob` with `inline` fallback.

The cell is **non-blocking** for the rest of the notebook: if training evidence cannot be obtained it prints a clear warning and continues.

In [ ]:
# ── Training-side learning signal ─────────────────────────────────────────
#
# TRAINING_MODE: "trainjob" | "inline" | "skip"
#   trainjob -> submit minimal KFT v2 TrainJob, wait, scrape reward/loss from logs
#   inline   -> run a tiny GRPO step in-process (CPU-safe)
#   skip     -> bypass entirely
#
# Defaults to "trainjob" with "inline" fallback.

import importlib, json, os, re, sys, time

TRAINING_MODE = os.getenv("TRAINING_MODE", "trainjob").lower()

_training_summary: dict = {}   # populated below; available for later cells


# ── helpers ──────────────────────────────────────────────────────────────────

def _scrape_spike_json(text: str):
    for line in text.splitlines():
        line = line.strip()
        if line.startswith("{") and "status" in line:
            try:
                obj = json.loads(line)
                if obj.get("status") == "ok":
                    return obj
            except json.JSONDecodeError:
                pass
    return None


def _print_training_summary(s: dict) -> None:
    print("\n=== TRAINING LEARNING SIGNAL ===")
    for key in [
        "model", "world_size", "train_n", "num_generations",
        "train_runtime_s",
        "train_reward_mean_first", "train_reward_mean_last",
        "train_loss_first", "train_loss_last",
    ]:
        if key in s:
            val = s[key]
            print(f"  {key}: {float(val):.4f}" if isinstance(val, float) else f"  {key}: {val}")
    r0, r1 = s.get("train_reward_mean_first"), s.get("train_reward_mean_last")
    l0, l1 = s.get("train_loss_first"), s.get("train_loss_last")
    if r0 is not None and r1 is not None:
        print(f"  reward_delta: {r1 - r0:+.4f}")
    if l0 is not None and l1 is not None:
        print(f"  loss_delta: {l1 - l0:+.4f}")
    print("================================\n")


# ── Mode: TrainJob (Kubeflow Trainer v2) ─────────────────────────────────────

def _run_trainjob_mode():
    try:
        from kubeflow.trainer import TrainerClient, CustomTrainer
    except ImportError:
        print("[training] kubeflow.trainer not installed; skipping TrainJob mode.")
        return None

    client = TrainerClient()
    try:
        runtimes = client.list_runtimes()
    except Exception as exc:
        print(f"[training] Cannot reach KFT v2 (list_runtimes failed: {exc}); skipping.")
        return None

    if not any(rt.name == "torch-distributed" for rt in runtimes):
        print("[training] torch-distributed runtime not found; skipping TrainJob mode.")
        return None

    # Self-contained GRPO function -- KFT cloudpickles this into the pod
    def _mini_grpo_train():
        import json, os, re, time, torch
        from datasets import load_dataset
        from transformers import AutoModelForCausalLM, AutoTokenizer
        from trl import GRPOConfig, GRPOTrainer

        RANK = int(os.environ.get("RANK", 0))
        WORLD_SIZE = int(os.environ.get("WORLD_SIZE", 1))
        LOCAL_RANK = int(os.environ.get("LOCAL_RANK", 0))
        USE_GPU = torch.cuda.is_available()
        if USE_GPU:
            torch.cuda.set_device(LOCAL_RANK)

        TRAIN_N, NUM_GEN = 32, 2
        PD_BS, GRAD_ACC = (1, 2) if not USE_GPU else (2, 1)
        MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
        dtype = (torch.bfloat16 if USE_GPU and torch.cuda.is_bf16_supported()
                 else (torch.float16 if USE_GPU else torch.float32))

        t0 = time.perf_counter()
        model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=dtype)
        tok = AutoTokenizer.from_pretrained(MODEL)
        tok.pad_token = tok.eos_token
        tok.padding_side = "left"

        def fmt(ex):
            return {
                "prompt": [
                    {"role": "system", "content": "Solve math. End with #### <number>."},
                    {"role": "user", "content": ex["question"]},
                ],
                "answer": ex["answer"],
            }

        ds = load_dataset("openai/gsm8k", "main", split="train").select(range(TRAIN_N)).map(fmt)

        def reward(completions, answer, **kw):
            out = []
            for c, a in zip(completions, answer):
                txt = c[0]["content"] if isinstance(c, list) else str(c)
                ref = re.search(r"####\s*([\d,]+)", a)
                pred = re.search(r"####\s*([\d,]+)", txt)
                rn = ref.group(1).replace(",", "") if ref else None
                pn = pred.group(1).replace(",", "") if pred else None
                out.append(1.0 if (rn and pn and rn == pn) else (0.1 if pred else 0.0))
            return out

        cfg = GRPOConfig(
            output_dir="/tmp/grpo-mini", num_train_epochs=1,
            per_device_train_batch_size=PD_BS, gradient_accumulation_steps=GRAD_ACC,
            num_generations=NUM_GEN, max_completion_length=64,
            generation_batch_size=NUM_GEN,  # TRL 1.4+: gen batch must be divisible by num_generations
            learning_rate=5e-6, beta=0.0,
            bf16=(USE_GPU and torch.cuda.is_bf16_supported()),
            fp16=bool(USE_GPU and not torch.cuda.is_bf16_supported()),
            logging_steps=1, save_strategy="no", report_to="none", eval_strategy="no",
        )
        trainer = GRPOTrainer(model=model, args=cfg, reward_funcs=reward,
                              train_dataset=ds, processing_class=tok)
        tr = trainer.train()
        wall = time.perf_counter() - t0
        metrics = getattr(tr, "metrics", {}) or {}
        logs = trainer.state.log_history
        rewards = [x.get("reward/mean") for x in logs if "reward/mean" in x]
        losses = [x.get("loss") for x in logs if x.get("loss") is not None]
        summary = {
            "status": "ok", "model": MODEL, "world_size": WORLD_SIZE,
            "train_n": TRAIN_N, "num_generations": NUM_GEN,
            "train_runtime_s": float(metrics.get("train_runtime", wall)),
            "train_reward_mean_first": float(rewards[0]) if rewards else None,
            "train_reward_mean_last": float(rewards[-1]) if rewards else None,
            "train_loss_first": float(losses[0]) if losses else None,
            "train_loss_last": float(losses[-1]) if losses else None,
        }
        if RANK == 0:
            print(json.dumps(summary))

    print("[training] Submitting minimal GRPO TrainJob (1 epoch, 32 samples) ...")
    try:
        job_name = client.train(
            trainer=CustomTrainer(
                func=_mini_grpo_train, num_nodes=1,
                resources_per_node={"cpu": 4, "memory": "16Gi"},
                packages_to_install=["trl", "datasets", "accelerate"],
            )
        )
    except Exception as exc:
        print(f"[training] TrainJob submission failed: {exc}; skipping.")
        return None

    print(f"[training] TrainJob submitted: {job_name}")
    deadline = time.time() + 900
    while time.time() < deadline:
        try:
            job = client.get_job(job_name)
            status_str = str(getattr(job, "status", "")).lower()
            print(f"[training] status={status_str}")
            if "succeed" in status_str or "complet" in status_str:
                break
            if "fail" in status_str or "error" in status_str:
                print("[training] TrainJob failed.")
                return None
        except Exception:
            pass
        time.sleep(20)

    try:
        log_text = "\n".join(client.get_job_logs(job_name, follow=False))
    except Exception as exc:
        print(f"[training] Could not fetch logs: {exc}")
        log_text = ""

    s = _scrape_spike_json(log_text)
    if s:
        print("[training] TrainJob PASSED -- learning signal captured.")
    else:
        print("[training] WARN: no SPIKE_SUMMARY_JSON found in logs.")
    return s


# ── Mode: Inline (in-process, CPU-safe) ──────────────────────────────────────

def _run_inline_mode():
    for pkg in ["trl", "datasets", "transformers", "torch"]:
        if importlib.util.find_spec(pkg) is None:
            print(f"[training] '{pkg}' not installed; skipping inline mode.")
            return None

    import re, time, torch
    from datasets import load_dataset
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from trl import GRPOConfig, GRPOTrainer

    USE_GPU = torch.cuda.is_available()
    TRAIN_N, NUM_GEN = 10, 2
    MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
    dtype = (torch.bfloat16 if USE_GPU and torch.cuda.is_bf16_supported()
             else (torch.float16 if USE_GPU else torch.float32))

    print(f"[training] Inline GRPO: {TRAIN_N} samples, 1 epoch, device={'GPU' if USE_GPU else 'CPU'}")

    t0 = time.perf_counter()
    try:
        model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=dtype)
        tok = AutoTokenizer.from_pretrained(MODEL)
    except Exception as exc:
        print(f"[training] Model load failed ({exc}); skipping.")
        return None

    tok.pad_token = tok.eos_token
    tok.padding_side = "left"

    def fmt(ex):
        return {
            "prompt": [
                {"role": "system", "content": "Solve math. End with #### <number>."},
                {"role": "user", "content": ex["question"]},
            ],
            "answer": ex["answer"],
        }

    ds = load_dataset("openai/gsm8k", "main", split="train").select(range(TRAIN_N)).map(fmt)

    def reward(completions, answer, **kw):
        out = []
        for c, a in zip(completions, answer):
            txt = c[0]["content"] if isinstance(c, list) else str(c)
            pred = re.search(r"####\s*([\d,]+)", txt)
            ref = re.search(r"####\s*([\d,]+)", a)
            pn = pred.group(1).replace(",", "") if pred else None
            rn = ref.group(1).replace(",", "") if ref else None
            out.append(1.0 if (rn and pn and rn == pn) else (0.1 if pred else 0.0))
        return out

    cfg = GRPOConfig(
        output_dir="/tmp/grpo-inline", num_train_epochs=1,
        per_device_train_batch_size=1, gradient_accumulation_steps=1,
        num_generations=NUM_GEN, max_completion_length=64,
        generation_batch_size=NUM_GEN,  # TRL 1.4+: gen batch must be divisible by num_generations
        learning_rate=5e-6, beta=0.0,
        bf16=False, fp16=False,  # CPU-safe; model already cast to correct dtype above
        logging_steps=1, save_strategy="no", report_to="none", eval_strategy="no",
    )

    try:
        trainer = GRPOTrainer(model=model, args=cfg, reward_funcs=reward,
                              train_dataset=ds, processing_class=tok)
        tr = trainer.train()
    except Exception as exc:
        print(f"[training] Inline GRPO training failed: {exc}")
        return None

    wall = time.perf_counter() - t0
    metrics = getattr(tr, "metrics", {}) or {}
    logs = trainer.state.log_history
    rewards = [x.get("reward/mean") for x in logs if "reward/mean" in x]
    losses = [x.get("loss") for x in logs if x.get("loss") is not None]
    return {
        "status": "ok", "mode": "inline", "model": MODEL,
        "train_n": TRAIN_N, "num_generations": NUM_GEN,
        "train_runtime_s": float(metrics.get("train_runtime", wall)),
        "train_reward_mean_first": float(rewards[0]) if rewards else None,
        "train_reward_mean_last": float(rewards[-1]) if rewards else None,
        "train_loss_first": float(losses[0]) if losses else None,
        "train_loss_last": float(losses[-1]) if losses else None,
    }


# ── Dispatch ──────────────────────────────────────────────────────────────────

if TRAINING_MODE == "skip":
    print("[training] TRAINING_MODE=skip -- bypassing learning-signal section.")
elif TRAINING_MODE == "inline":
    _training_summary = _run_inline_mode() or {}
elif TRAINING_MODE == "trainjob":
    _training_summary = _run_trainjob_mode() or {}
    if not _training_summary:
        print("[training] TrainJob mode unavailable; falling back to inline GRPO.")
        _training_summary = _run_inline_mode() or {}
else:
    print(f"[training] Unknown TRAINING_MODE={TRAINING_MODE!r}; falling back to inline.")
    _training_summary = _run_inline_mode() or {}

if _training_summary.get("status") == "ok":
    _print_training_summary(_training_summary)
    print("[training] Section PASS -- learning signal confirmed.")
elif TRAINING_MODE != "skip":
    print("[training] WARNING: Could not obtain training learning signal in this environment.")
    print("           Set TRAINING_MODE=skip to suppress, or check cluster connectivity.")

## 2) Baseline BEFORE weight sync

This captures rollout performance before validation/sync.


In [ ]:
def run_baseline(tag: str):
    out_dir = OUTPUT_ROOT / tag
    out_dir.mkdir(parents=True, exist_ok=True)

    cmd = (
        f"{shlex.quote(str(SCRIPT_BASELINE))} "
        f"--namespace {shlex.quote(NAMESPACE)} "
        f"--notebook-pod {shlex.quote(NOTEBOOK_POD)} "
        f"--notebook-container {shlex.quote(NOTEBOOK_CONTAINER)} "
        f"--vllm-service {shlex.quote(VLLM_SERVICE)} "
        f"--vllm-selector {shlex.quote(VLLM_SELECTOR)} "
        f"--model-name {shlex.quote(MODEL_NAME)} "
        f"--requests {int(BASELINE_REQUESTS)} "
        f"--max-tokens {int(BASELINE_MAX_TOKENS)} "
        f"--output-dir {shlex.quote(str(out_dir))}"
    )

    last_proc = None
    for attempt in range(1, 3):
        last_proc = run(cmd, check=False)
        if last_proc.returncode == 0:
            break

        combined = f"{last_proc.stdout or ''}\n{last_proc.stderr or ''}"
        transient_not_found = 'Error from server (NotFound): pods "' in combined

        if attempt < 2 and transient_not_found:
            print('Baseline probe hit transient pod-not-found; retrying after rollout pod refresh...')
            run(
                f'oc get pods -n {shlex.quote(NAMESPACE)} -l {shlex.quote(VLLM_SELECTOR)} -o wide',
                check=False,
            )
            time.sleep(5)
            continue

        raise RuntimeError(f'Baseline capture failed (attempt={attempt}): {cmd}')

    if last_proc is None or last_proc.returncode != 0:
        raise RuntimeError(f'Baseline capture did not succeed: {cmd}')

    raw_path = latest_file(out_dir, 'vllm-baseline-raw-*.json')
    summary_path = latest_file(out_dir, 'vllm-baseline-summary-*.md')
    data = json.loads(raw_path.read_text())
    metrics = extract_metrics(data)

    print('\n=== BASELINE', tag.upper(), '===')
    print('raw:', raw_path)
    print('summary:', summary_path)
    print(json.dumps(metrics, indent=2))

    return {
        'tag': tag,
        'out_dir': out_dir,
        'raw': raw_path,
        'summary': summary_path,
        'payload': data,
        'metrics': metrics,
    }

BEFORE = run_baseline('before')

## 3) Deterministic weight-sync validation

This proves unsynced -> synced transition and prints sync evidence directly in output.


In [ ]:
skip_flag = '--skip-restart' if VALIDATION_SKIP_RESTART else ''
validate_cmd = (
    f"{shlex.quote(str(SCRIPT_VALIDATE))} "
    f"--namespace {shlex.quote(NAMESPACE)} "
    f"--notebook-pod {shlex.quote(NOTEBOOK_POD)} "
    f"--notebook-container {shlex.quote(NOTEBOOK_CONTAINER)} "
    f"--vllm-deployment {shlex.quote(VLLM_DEPLOYMENT)} "
    f"--vllm-service {shlex.quote(VLLM_SERVICE)} "
    f"--vllm-selector {shlex.quote(VLLM_SELECTOR)} "
    f"--model-name {shlex.quote(MODEL_NAME)} "
    f"--attempts {int(VALIDATION_ATTEMPTS)} "
    f"{skip_flag}"
).strip()

run(validate_cmd)
print('\nWeight sync validation: PASS')


## 4) Baseline AFTER weight sync

This captures post-sync performance using the same parameters.


In [ ]:
AFTER = run_baseline('after')


In [ ]:
print('=== PERFORMANCE COMPARISON (after vs before) ===')
rows = [
    'latency_ms_p50',
    'latency_ms_p95',
    'ttft_ms_p50',
    'ttft_ms_p95',
    'throughput_rps',
    'token_throughput_tokens_per_s',
    'error_rate',
]

for k in rows:
    b = BEFORE['metrics'].get(k)
    a = AFTER['metrics'].get(k)
    d = pct_delta(b, a)
    delta = 'n/a' if d is None else f'{d:+.2f}%'
    print(f'- {k}: before={b} | after={a} | delta={delta}')


## 5) Rollout service health + control-plane checks

This cell validates health and control-plane endpoints on the rollout service.


In [ ]:
import urllib.error
import urllib.request

service_host = f"{VLLM_SERVICE}.{NAMESPACE}.svc.cluster.local"
service_root = f"http://{service_host}:8000"
print('Rollout service root:', service_root)


def _curl_fallback(path: str, method: str):
    url = f"{service_root}{path}"
    method_arg = '' if method == 'GET' else f'-X {method} '
    cmd = (
        f"oc exec -n {shlex.quote(NAMESPACE)} {shlex.quote(NOTEBOOK_POD)} "
        f"-c {shlex.quote(NOTEBOOK_CONTAINER)} -- "
        f"curl -sS -w '\n%{{http_code}}' {method_arg}{shlex.quote(url)}"
    )
    proc = run(cmd, check=False)
    out = (proc.stdout or '').strip()
    lines = out.splitlines() if out else []
    status = None
    body = ''
    if lines:
        maybe_code = lines[-1].strip()
        if maybe_code.isdigit():
            status = int(maybe_code)
            body = '\n'.join(lines[:-1])
        else:
            body = out
    return status, body, proc.returncode


def check_endpoint(path: str, method: str = 'GET'):
    url = f"{service_root}{path}"
    try:
        req = urllib.request.Request(url=url, method=method)
        with urllib.request.urlopen(req, timeout=30) as resp:
            body = resp.read().decode('utf-8', errors='replace')
            print(f"{method} {path}: PASS (direct) status={resp.status}")
            print(body[:300])
            return True
    except (urllib.error.URLError, urllib.error.HTTPError, TimeoutError, OSError) as exc:
        print(f"{method} {path}: direct request failed ({exc}); trying oc exec fallback")

    status, body, rc = _curl_fallback(path, method)
    if rc == 0 and status is not None and 200 <= status < 300:
        print(f"{method} {path}: PASS (oc exec fallback) status={status}")
        print(body[:300])
        return True

    print(f"{method} {path}: FAIL (rc={rc}, status={status})")
    if body:
        print(body[:500])
    return False


checks = [
    ('GET', '/health'),
    ('GET', '/get_world_size'),
    ('POST', '/pause'),
    ('POST', '/resume'),
]

failures = []
for method, path in checks:
    ok = check_endpoint(path=path, method=method)
    if not ok:
        failures.append(f"{method} {path}")

if failures:
    raise RuntimeError('Rollout service checks failed: ' + ', '.join(failures))

print('Rollout service health/control-plane checks: PASS')


## Final interpretation

Use this notebook as a focused training + rollout validation package:

- **Step 1 (Training signal)**: If `[training] Section PASS` appears, a GRPO step ran and emitted runtime + reward/loss markers. `TRAINING_MODE=trainjob` runs a real cluster `TrainJob`; `TRAINING_MODE=inline` runs a local micro-GRPO step (CPU-safe). Set `TRAINING_MODE=skip` to bypass.
- **Step 2 (Baseline before)**: Rollout latency/throughput captured before weight sync.
- **Step 3 (Weight sync)**: If `validate_cold_start_weight_sync.sh` exits 0 and the sync probe converges, the unsynced → synced transition is confirmed.
- **Step 4 (Baseline after)**: Rollout performance captured after sync. Compare against Step 2 delta.
- **Step 5 (Control-plane)**: `/health`, `/get_world_size`, `/pause`, `/resume` all return 2xx — rollout control-plane is healthy.

**Environment flags:**

| Variable | Default | Purpose |
|----------|---------|---------|
| `TRAINING_MODE` | `trainjob` | `trainjob`, `inline`, or `skip` |
| `OFFLINE_DEMO` | `false` | Set `true` to bypass cluster calls; all rollout sections print `[OFFLINE_DEMO]` markers |
| `GRPO_TRAINER_ROOT` | auto-detect | Explicit path to the grpo-trainer repo root |

Run artifacts are written under:

`research/grpo-trainer/runs_log/notebook-final-validation-<timestamp>/`
